código feito por mim mas usando os exemplos da aula como base.

inicialmente são feitas as importações necessárias para código funcionar.

In [ ]:
from enum import auto, IntFlag
from typing import Any
import re

from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    model_validator,
    SecretStr,
    ValidationError,
)


verificação adicional criada para o nome, exige que tenha no mínimo duas caracteres. dentro da validação em si eu adiciono também a condição de ter no mínimo duas letras distinas.

In [2]:
NOME_VALIDO_TAMANHO = re.compile(r"^[a-zA-Z]{2,}$")

a ideia era simular um mini sistema escolar, onde se tem Aluno, Professor e Diretor, por isso existem essas 3 categorias para o usuário, com o default sendo Aluno. Na class Usário, são definidos os atributos de cada usuário, usando o BaseModel com suas verificações automáticas e o model_validor para fazer as verificações adicionais. Como verificações adicionais, adicionei a que se refere ao tamanho do nome, precisa ter pelo menos duas letras e pelo menos duas distintas.

In [3]:
class Categoria (IntFlag):
    Aluno  = 1
    Professor = 2
    Diretor = 4

class Usuario (BaseModel):
    nome: str = Field(examples=["Adriam", "Felipe"])
    email: EmailStr = Field(
        examples = ["adriam@fastcamp.com"],
        frozen=True,
    )
    senha: SecretStr = Field(examples=["lamiafastcampadriam"])
    matricula:  int = Field(examples=["202500000"])
    categoria: Categoria = Field(default=1)

    @model_validator(mode ="before")
    @classmethod
    def validate_all(cls, v:dict[str, Any]) -> dict[str, Any]:
        nome = v.get("nome", "")
        if not NOME_VALIDO_TAMANHO.match(nome):
            raise ValueError(
                "Um nome válido precisa ter no mínimo duas letras"
            )
        st = set(nome.lower())  
        
        if len(st) < 2:
            raise ValueError(
                "Nome inválido, precisa ter no mínimo duas letras distintas."
            )

        return v

Função que requisita a validação dos dados.

In [4]:
def validacao (data: dict[str, Any]) -> None:
    try:
        usuario = Usuario.model_validate(data)
        print(usuario)
    except ValidationError as erro:
        print("Usuário inválido")
        for erros in erro.errors():
            print(erros)

Função main, onde são executados os testes.

In [ ]:
def main() -> None:
    testes = dict(
        correto={
            "nome":"Adriam",
            "email":"adriam@lamia.com",
            "senha":"1234abcd",
            "matricula": 20250000,
        },
        matricula_errada ={
            "nome":"Adriam",
            "email":"adriam@lamia.com",
            "senha":"1234abcd",
            "matricula": "2025000c",
        },
        email_errado ={
            "nome":"Adriam",
            "email":"adriamlamia.com",
            "senha":"1234abcd",
            "matricula": 20250000,
            "categoria": 4,
        }, 
        nome_errado={
            "nome":"Aaaaaa",
            "email":"adriam@lamia.com",
            "senha":"1234abcd",
            "matricula": 20250000,
            "categoria": 2,
        },
    )

    for nome_teste, dados in testes.items():
        print(nome_teste)
        validacao(dados)
        print()


if __name__ == "__main__":
    main()

correto
nome='Adriam' email='adriam@lamia.com' senha=SecretStr('**********') matricula=20250000 categoria=1

matricula_errada
Usuário inválido
{'type': 'int_parsing', 'loc': ('matricula',), 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': '2025000c', 'url': 'https://errors.pydantic.dev/2.12/v/int_parsing'}

email_errado
Usuário inválido
{'type': 'value_error', 'loc': ('email',), 'msg': 'value is not a valid email address: An email address must have an @-sign.', 'input': 'adriamlamia.com', 'ctx': {'reason': 'An email address must have an @-sign.'}}

nome_errado
Usuário inválido
{'type': 'value_error', 'loc': (), 'msg': 'Value error, Nome inválido, precisa ter no mínimo duas letras distintas.', 'input': {'nome': 'Aaaaaa', 'email': 'adriam@lamia.com', 'senha': '1234abcd', 'matricula': 20250000}, 'ctx': {'error': ValueError('Nome inválido, precisa ter no mínimo duas letras distintas.')}, 'url': 'https://errors.pydantic.dev/2.12/v/value_error'}

